# 한국마사회 개별면접 리포트 생성 (v3.1)

## 실행 순서

| 단계 | 하는 일 | 결과물 |
|---|---|---|
| STEP 1 | 자기소개서 **문항별 요약** + **맞춤형 면접질문** 생성 | 결과 엑셀 (`면접질문` · `문항요약` 두 시트) |
| STEP 2 | 리포트 HTML 생성 | `면접리포트_*.html`, `전체리포트.html` |

경로·API 키·리포트 제목은 **셀을 실행하면 입력창이 뜹니다.** 코드를 직접 고칠 필요가 없습니다.

- 경로는 파일 탐색기에서 파일 우클릭 → **경로로 복사** 후 붙여넣으면 됩니다. 따옴표는 자동으로 제거됩니다.
- 대괄호로 기본값이 표시된 항목은 그대로 쓰려면 **Enter만** 누르세요.
- STEP 1은 이미 생성된 지원자를 건너뛰므로, 중간에 끊겨도 다시 실행하면 이어서 진행됩니다.

## 평가요소 구성

주질문은 **정신자세 · 전문지식 · 품행 · 발전가능성** 4개이며, 자기소개서 엑셀의
`평가요소` 시트(`모집분야 | 평가요소 | 내용`)를 참조해 지원자별로 맞춤 생성됩니다.

**발표력**은 자기소개서로 검증할 수 없는 요소이므로 질문 대신
**행동지표 관찰표**(긍정요소/부정요소)가 리포트에 고정 삽입됩니다.

## 보안 안내

기존 질문생성 노트북에 OpenAI API 키가 평문으로 저장되어 있었습니다.
해당 키는 폐기(revoke)하고 새 키를 발급받으세요.

키를 매번 입력하기 번거로우면 환경변수에 등록해 두면 STEP 1이 자동으로 사용합니다.

```
Windows CMD:  setx OPENAI_API_KEY "sk-..."
```
등록 후 Jupyter 커널을 재시작해야 반영됩니다.


In [ ]:
!pip install openai openpyxl

---
## STEP 1 — 자기소개서 요약 + 면접질문 생성

실행하면 순서대로 입력받습니다.

1. 자기소개서 엑셀 경로
2. 결과 저장 경로 *(Enter = 자기소개서 파일명 뒤에 `_면접질문생성결과` 붙여 자동 생성)*
3. 모델명 *(Enter = gpt-5.6)*
4. OpenAI API 키 *(환경변수가 있으면 건너뜀 / 입력해도 화면에 보이지 않음)*
5. 기존 결과 재사용 여부 *(Enter = 예)*

지원자 1명당 API 호출 1회로 요약과 질문을 함께 생성합니다.


In [ ]:
# -*- coding: utf-8 -*-
"""
[STEP 1] 자기소개서 문항별 요약 + 맞춤형 면접질문 생성

입력 : 자기소개서 엑셀 (시트 "자기소개서", "평가요소")
출력 : 결과 엑셀 1개
        - 시트 "면접질문"  : 면접번호|이름|모집분야|평가요소|주질문|근거구절|질문이유|보조질문
        - 시트 "문항요약"  : 면접번호|문항번호|문항질문|핵심요약|요약항목|작성글자수|공백제외글자수

* 경로와 API 키는 실행 시 입력창으로 받습니다.
* 이미 처리된 지원자는 건너뛰므로 중단 후 재실행해도 이어서 진행됩니다.
"""

import json
import os
import re
import time
from getpass import getpass

import openpyxl
from openai import OpenAI


# ======================================================================
# 입력 도우미
# ======================================================================
def ask(label, default=""):
    """경로/문자열 입력. 앞뒤 따옴표는 자동 제거(Windows '경로로 복사' 대응)."""
    if default:
        raw = input(f"{label}\n  [Enter = {default}]\n > ")
    else:
        raw = input(f"{label}\n > ")
    value = (raw or "").strip() or default
    return value.strip().strip('"').strip("'")


def ask_yes_no(label, default=True):
    hint = "Y/n" if default else "y/N"
    raw = input(f"{label} ({hint})\n > ").strip().lower()
    if not raw:
        return default
    return raw in ("y", "yes", "예", "ㅇ")


# ======================================================================
# 1) 설정 입력
# ======================================================================
print("=" * 66)
print(" STEP 1 · 자기소개서 요약 + 면접질문 생성")
print("=" * 66)

INTRO_PATH = ask("① 자기소개서 엑셀 파일 경로를 입력하세요")

while not os.path.exists(INTRO_PATH):
    print(f"   ! 파일을 찾을 수 없습니다: {INTRO_PATH}")
    INTRO_PATH = ask("① 자기소개서 엑셀 파일 경로를 다시 입력하세요")

_base, _ext = os.path.splitext(INTRO_PATH)
_default_out = f"{_base}_면접질문생성결과.xlsx"

RESULT_PATH = ask("② 결과를 저장할 엑셀 파일 경로", _default_out)

MODEL = ask("③ 사용할 모델명", "gpt-5.6")

_env_key = os.getenv("OPENAI_API_KEY")
if _env_key:
    print("\n④ OPENAI_API_KEY 환경변수를 사용합니다.")
    API_KEY = _env_key
else:
    print("\n④ OpenAI API 키를 입력하세요 (입력 내용은 화면에 표시되지 않습니다)")
    API_KEY = getpass(" > ").strip()

if not API_KEY:
    raise RuntimeError("API 키가 입력되지 않았습니다.")

REUSE = ask_yes_no("\n⑤ 기존 결과 파일에 이미 생성된 지원자는 건너뛸까요?", True)

client = OpenAI(api_key=API_KEY)

QUESTION_SHEET = "면접질문"
SUMMARY_SHEET = "문항요약"

QUESTION_HEADERS = ["면접번호", "이름", "모집분야", "평가요소",
                    "주질문", "근거구절", "질문이유", "보조질문"]
SUMMARY_HEADERS = ["면접번호", "문항번호", "문항질문",
                   "핵심요약", "요약항목", "작성글자수", "공백제외글자수"]

# 주질문 평가요소 (발표력은 리포트에서 행동지표 표로 고정 제공하므로 제외)
FACTORS = ["정신자세", "전문지식", "품행", "발전가능성"]


# ======================================================================
# 2) 데이터 로드
# ======================================================================
def load_intro(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["자기소개서"]

    data = {}
    order = []
    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[0] is None:
            continue
        exam_no, name, field, status, question, answer = row[:6]
        if exam_no not in data:
            data[exam_no] = {
                "면접번호": exam_no,
                "이름": name,
                "모집분야": field,
                "전형단계": status,
                "qa": [],
            }
            order.append(exam_no)
        data[exam_no]["qa"].append({
            "question": str(question or ""),
            "answer": str(answer or ""),
        })
    return [data[k] for k in order]


def load_eval_criteria(path):
    """'평가요소' 시트 -> {모집분야: {평가요소: [내용, ...]}}
    시트가 없거나 형식이 달라도 오류 없이 빈 dict 반환."""
    wb = openpyxl.load_workbook(path, data_only=True)
    if "평가요소" not in wb.sheetnames:
        print("   [안내] '평가요소' 시트가 없어 기본 정의로 질문을 생성합니다.")
        return {}

    ws = wb["평가요소"]
    info = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[0] is None:
            continue
        field = row[0]
        factor = row[1] if len(row) > 1 else None
        content = row[2] if len(row) > 2 else None
        info.setdefault(field, {}).setdefault(factor, [])
        if content:
            info[field][factor].append(str(content))
    return info


def build_eval_text(field, eval_info):
    if field not in eval_info:
        return "(제공된 평가요소 정의 없음 - 일반적인 공공기관 면접 기준을 적용할 것)"
    lines = []
    for factor, contents in eval_info[field].items():
        lines.append(f"[{factor}]")
        for c in contents:
            lines.append(f"- {c}")
    return "\n".join(lines)


# ======================================================================
# 3) 기존 결과 로드 (이어서 실행)
# ======================================================================
def load_existing(path):
    if not (REUSE and os.path.exists(path)):
        return {}, {}

    wb = openpyxl.load_workbook(path, data_only=True)
    q_rows, s_rows = {}, {}

    if QUESTION_SHEET in wb.sheetnames:
        for row in wb[QUESTION_SHEET].iter_rows(min_row=2, values_only=True):
            if row[0] is None:
                continue
            q_rows.setdefault(row[0], []).append(list(row))

    if SUMMARY_SHEET in wb.sheetnames:
        for row in wb[SUMMARY_SHEET].iter_rows(min_row=2, values_only=True):
            if row[0] is None:
                continue
            s_rows.setdefault(row[0], []).append(list(row))

    return q_rows, s_rows


# ======================================================================
# 4) GPT 호출
# ======================================================================
SYSTEM_PROMPT = """
당신은 공공기관 블라인드 채용 면접관이다.
반드시 JSON만 출력한다. 마크다운 코드블록이나 설명 문장은 출력하지 않는다.

출력 형식:

{
 "summaries":[
   {"no":1, "headline":"", "points":["",""]}
 ],
 "main_questions":[
   {"factor":"", "question":"", "source_quote":"", "reason":"",
    "sub_questions":["","",""]}
 ]
}

■ summaries (자기소개서 문항별 요약) 규칙
- 입력에 제시된 모든 문항을 빠짐없이 요약하며, "no"는 문항 번호를 그대로 사용한다.
- "headline"은 해당 문항 답변의 핵심을 한 문장(공백 포함 40자 이내)으로 압축한다.
- "points"는 핵심 내용 2~3개. 각 항목 50자 이내, 명사형으로 끝맺는다.
- 지원자가 실제로 작성한 내용만 요약한다. 없는 사실을 추론하거나 덧붙이지 않는다.
- 우수함/부족함 등 평가·판단 표현을 쓰지 않는다. 사실 요약만 한다.
- 답변이 비어 있으면 headline은 "미작성", points는 빈 배열로 둔다.

■ main_questions (맞춤형 면접질문) 규칙
- 주질문은 정확히 4개를 생성하며, 순서와 평가요소를 아래와 같이 고정한다.
  1) factor="정신자세"    2) factor="전문지식"
  3) factor="품행"        4) factor="발전가능성"
- 각 평가요소의 정의와 질문 방향은 사용자 메시지의 "평가요소" 항목을 참조하여,
  해당 지원자에게 맞춘 개별 질문으로 구성한다.
- 각 주질문마다 보조질문 3개를 생성한다. 보조질문은 해당 평가요소를 더 깊이
  검증하는 꼬리질문(구체적 상황, 본인의 행동, 결과, 교훈 등)으로 구성한다.
- 경험기반면접(BEI) 방식으로 만들며, 현장에서 읽기 쉽도록 간결하게 작성한다.
- "reason"에는 해당 질문이 어떤 평가요소를 어떻게 검증하려는 것인지 명시한다.
- "source_quote"는 질문의 근거가 된 자기소개서 답변 속 문장을 원문 그대로
  1~2문장 이내로 짧게 인용한다. 요약하거나 새로 만들어내지 않는다.
- 자기소개서에 직접적인 근거 문장이 없으면 source_quote를 빈 문자열("")로 둔다.

■ 공통 규칙 (블라인드 채용)
- 이름, 출신학교, 출신지역, 나이, 성별, 가족관계, 신체조건 등 개인 식별정보는
  요약과 질문 어디에도 포함하지 않는다.
- 위 정보를 묻거나 유추하게 만드는 질문을 생성하지 않는다.
"""


def build_user_prompt(candidate, eval_text):
    intro_text = ""
    for idx, qa in enumerate(candidate["qa"], start=1):
        intro_text += f"""
[문항 {idx}]
질문:
{qa['question']}

답변:
{qa['answer']}
"""

    return f"""
모집분야:
{candidate['모집분야']}

평가요소:

{eval_text}

자기소개서:
{intro_text}
"""


def parse_json(raw):
    text = str(raw or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
        text = re.sub(r"```\s*$", "", text).strip()
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1:
        text = text[start:end + 1]
    return json.loads(text)


def call_gpt(candidate, eval_text, retries=2):
    last_err = None
    for attempt in range(retries + 1):
        try:
            response = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(candidate, eval_text)},
                ],
            )
            return parse_json(response.output_text)
        except json.JSONDecodeError as e:
            last_err = e
            if attempt < retries:
                print("(JSON 파싱 실패, 재시도)", end=" ")
                time.sleep(2)
        except Exception as e:
            last_err = e
            if attempt < retries:
                print("(호출 실패, 재시도)", end=" ")
                time.sleep(3)
    raise last_err


# ======================================================================
# 5) 결과 행 변환
# ======================================================================
def count_chars(text):
    t = str(text or "")
    return len(t), len(re.sub(r"\s", "", t))


def to_question_rows(candidate, data):
    rows = []
    mqs = data.get("main_questions", []) or []
    for i, q in enumerate(mqs):
        factor = str(q.get("factor") or "").strip()
        if factor not in FACTORS:
            factor = FACTORS[i] if i < len(FACTORS) else f"주질문 {i + 1}"
        subs = q.get("sub_questions", []) or []
        rows.append([
            candidate["면접번호"],
            candidate["이름"],
            candidate["모집분야"],
            factor,
            str(q.get("question", "")),
            str(q.get("source_quote", "")),
            str(q.get("reason", "")),
            "\n".join(str(s) for s in subs),
        ])
    return rows


def to_summary_rows(candidate, data):
    by_no = {}
    for s in (data.get("summaries", []) or []):
        try:
            by_no[int(s.get("no", 0))] = s
        except (TypeError, ValueError):
            continue

    rows = []
    for q_no, qa in enumerate(candidate["qa"], start=1):
        s = by_no.get(q_no, {})
        points = s.get("points", []) or []
        total, nospace = count_chars(qa["answer"])
        rows.append([
            candidate["면접번호"],
            q_no,
            qa["question"],
            str(s.get("headline", "")),
            "\n".join(str(p) for p in points),
            total,
            nospace,
        ])
    return rows


# ======================================================================
# 6) 저장
# ======================================================================
def save_result(path, question_rows, summary_rows):
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    ws_q = wb.create_sheet(QUESTION_SHEET)
    ws_q.append(QUESTION_HEADERS)
    for r in question_rows:
        ws_q.append(r)
    for col, width in zip("ABCDEFGH", [12, 10, 18, 12, 55, 55, 55, 70]):
        ws_q.column_dimensions[col].width = width

    ws_s = wb.create_sheet(SUMMARY_SHEET)
    ws_s.append(SUMMARY_HEADERS)
    for r in summary_rows:
        ws_s.append(r)
    for col, width in zip("ABCDEFG", [12, 10, 50, 45, 70, 12, 14]):
        ws_s.column_dimensions[col].width = width

    wb.save(path)


# ======================================================================
# 7) 실행
# ======================================================================
def main():
    candidates = load_intro(INTRO_PATH)
    eval_info = load_eval_criteria(INTRO_PATH)
    old_q, old_s = load_existing(RESULT_PATH)

    question_rows, summary_rows = [], []
    total = len(candidates)
    print(f"\n총 {total}명 처리 시작\n" + "-" * 66)

    for idx, candidate in enumerate(candidates, start=1):
        exam_no = candidate["면접번호"]

        if exam_no in old_q and exam_no in old_s:
            print(f"[{idx}/{total}] {exam_no} · 기존 결과 재사용")
            question_rows.extend(old_q[exam_no])
            summary_rows.extend(old_s[exam_no])
            continue

        print(f"[{idx}/{total}] {exam_no} · 생성중 ...", end=" ")
        eval_text = build_eval_text(candidate["모집분야"], eval_info)

        try:
            data = call_gpt(candidate, eval_text)
            q_rows = to_question_rows(candidate, data)
            s_rows = to_summary_rows(candidate, data)

            question_rows.extend(q_rows)
            summary_rows.extend(s_rows)
            print(f"완료 (질문 {len(q_rows)}개 / 요약 {len(s_rows)}문항)")

        except Exception as e:
            print("실패:", e)
            # 실패해도 요약 시트에 글자수는 남겨 리포트가 끊기지 않게 함
            for q_no, qa in enumerate(candidate["qa"], start=1):
                t, n = count_chars(qa["answer"])
                summary_rows.append([exam_no, q_no, qa["question"], "", "", t, n])

        time.sleep(1)

    save_result(RESULT_PATH, question_rows, summary_rows)

    print("-" * 66)
    print(f"저장 완료 → {RESULT_PATH}")
    print(f"  · '{QUESTION_SHEET}' {len(question_rows)}행")
    print(f"  · '{SUMMARY_SHEET}' {len(summary_rows)}행")
    print("\nSTEP 2 셀에서 이 경로를 그대로 사용하세요.")


main()


---
## STEP 2 — 리포트 HTML 생성

실행하면 순서대로 입력받습니다.

1. 자기소개서 엑셀 경로 *(STEP 1을 방금 돌렸다면 Enter로 재사용)*
2. STEP 1 결과 엑셀 경로 *(동일)*
3. 저장 폴더 *(Enter = 자기소개서 파일이 있는 폴더의 `개인별리포트`)*
4. **리포트 제목** *(Enter = 기본 제목)*
5. 로고 경로 *(Enter = 내장된 한국마사회 로고 사용)*

STEP 1을 건너뛰어도 리포트는 생성됩니다. 이 경우 요약은 자기소개서 원문에서
자동 추출한 내용으로 대체되며, 글자수는 원본에서 직접 계산하므로 항상 정확합니다.

리포트 상단 안내 문구는 코드 안의 `NOTICE_TEXT` 값을 고치면 바뀝니다.

### PDF로 저장하기
브라우저에서 HTML을 열고 `Ctrl+P` → 대상 **PDF로 저장** → 용지 **A4**, 배율 **기본**

화면과 동일한 색상·레이아웃으로 출력되며, **배경 그래픽** 옵션은 켜든 끄든 결과가 같습니다.
질문 블록·요약표 행·발표력 표는 페이지 경계에서 잘리지 않고 통째로 다음 장으로 넘어갑니다.


In [1]:
# -*- coding: utf-8 -*-
"""
[STEP 2] 지원자별 면접 리포트 HTML 생성 (v3)

입력:
  - 자기소개서 원본 엑셀 (시트 "자기소개서")
  - STEP 1 결과 엑셀 (시트 "면접질문", "문항요약")

출력:
  - 지원자별 개별 HTML 리포트 (면접리포트_<면접번호>.html)
  - 통합 리포트 (전체리포트.html) : 좌측 목록 클릭으로 지원자 전환

특징:
  - 리포트 상단 = 자기소개서 문항별 요약 (작성 글자수 포함)
  - 한국마사회 로고 내장(base64) → 별도 이미지 파일 불필요
  - 발표력 행동지표 표 고정 삽입, 인쇄(PDF) 레이아웃 대응
"""

import os
import re
import html
import base64
import openpyxl


# ======================================================================
# 입력 도우미
# ======================================================================
def ask(label, default=""):
    if default:
        raw = input(f"{label}\n  [Enter = {default}]\n > ")
    else:
        raw = input(f"{label}\n > ")
    value = (raw or "").strip() or default
    return value.strip().strip('"').strip("'")


# ======================================================================
# 설정 입력
# ======================================================================
print("=" * 66)
print(" STEP 2 · 개별 면접리포트 HTML 생성")
print("=" * 66)

# STEP 1을 같은 세션에서 실행했다면 그 경로를 기본값으로 제안
_d_intro = globals().get("INTRO_PATH", "")
_d_result = globals().get("RESULT_PATH", "")

INTRO_PATH = ask("① 자기소개서 엑셀 파일 경로", _d_intro)
while not os.path.exists(INTRO_PATH):
    print(f"   ! 파일을 찾을 수 없습니다: {INTRO_PATH}")
    INTRO_PATH = ask("① 자기소개서 엑셀 파일 경로를 다시 입력하세요")

INTERVIEW_PATH = ask("② 면접질문 생성결과 엑셀 파일 경로 (STEP 1 결과)", _d_result)
while not os.path.exists(INTERVIEW_PATH):
    print(f"   ! 파일을 찾을 수 없습니다: {INTERVIEW_PATH}")
    INTERVIEW_PATH = ask("② 면접질문 생성결과 엑셀 파일 경로를 다시 입력하세요")

OUTPUT_DIR = ask(
    "③ 리포트를 저장할 폴더",
    os.path.join(os.path.dirname(INTRO_PATH), "개인별리포트"),
)

REPORT_TITLE = ask(
    "④ 리포트 제목",
    "2026년 한국마사회 전임직·위촉직채용 개별면접질문",
)

LOGO_OVERRIDE_PATH = ask(
    "⑤ 로고 이미지 경로 (한국마사회 로고를 쓰려면 그냥 Enter)",
    "",
) or None

print()

# ======================================================================
# 리포트 상단 안내 문구 (콜아웃)
#   문구를 바꾸려면 아래 텍스트만 수정하세요. 줄바꿈도 그대로 반영됩니다.
# ======================================================================
NOTICE_TEXT = (
    "본 리포트는 지원자의 자기소개서 문항의 핵심 내용을 요약하고, "
    "이를 바탕으로 지원자의 경험과 역량을 확인할 수 있는 면접질문으로 "
    "구성되어 있습니다. 지원자의 강점과 역량을 평가하는데 활용해주시기 바라며, "
    "자기소개서의 내용과 면접 답변을 종합적으로 검토하여 지원자의 역량과 "
    "적합성을 판단해주시기 바랍니다."
)

_KRA_LOGO_B64 = (
    "PHN2ZyB3aWR0aD0iMTM3IiBoZWlnaHQ9IjcwIiB2aWV3Qm94PSIwIDAgMTM3IDcwIiBmaWxsPSJub25lIiB4bWxucz0iaHR0cDov"
    "L3d3dy53My5vcmcvMjAwMC9zdmciPgo8ZyBpZD0ibG9nb19rcmEiIGNsaXAtcGF0aD0idXJsKCNjbGlwMF8xMDFfNjg4KSI+Cjxw"
    "YXRoIGlkPSJWZWN0b3IiIGQ9Ik00Ljg4MjI2IDE3Ljk0OTdIMFY0MS40OTUzSDQuODgyMjZWMTcuOTQ5N1oiIGZpbGw9IiMxNDQ3"
    "ODIiLz4KPGcgaWQ9Ikdyb3VwIj4KPHBhdGggaWQ9IlZlY3Rvcl8yIiBkPSJNMjguOTU2OSAxNy45NDk3TDEyLjE3NzYgMjguOTAy"
    "NkMxMS44MzI5IDI5LjEzMzUgMTEuNjQwNSAyOS40MiAxMS42NDA1IDI5LjcxNDVDMTEuNjQwNSAzMC4wMDkxIDExLjgzMjkgMzAu"
    "MzAzNiAxMi4xNzc2IDMwLjUyNjVMMjguOTU2OSA0MS40Nzk0SDIwLjgxOThMNy43NjgzNiAzMi45NTQzQzYuNDQ1NTggMzIuMDk0"
    "NiA1LjcyNDA2IDMwLjk0ODMgNS43MjQwNiAyOS43MTQ1QzUuNzI0MDYgMjguNDgwOCA2LjQ1MzU5IDI3LjMyNjYgNy43NjgzNiAy"
    "Ni40NzQ4TDIwLjgxOTggMTcuOTQ5N0gyOC45NTY5WiIgZmlsbD0iIzE0NDc4MiIvPgo8cGF0aCBpZD0iVmVjdG9yXzMiIGQ9Ik02"
    "MS4wNDg0IDI0LjY5OThDNjEuMDQ4NCAyMC40NDEyIDU2Ljk2NzggMTcuOTQ5NyA1MS41NTY0IDE3Ljk0OTdIMzAuMTExM1YyMS4y"
    "OTI5SDUwLjUxNDJDNTMuNDY0NCAyMS4yOTI5IDU1Ljg2MTUgMjIuMzkxNCA1NS44NjE1IDI0LjY5OThDNTUuODYxNSAyNy4wMDgy"
    "IDUzLjQ2NDQgMjguMTA2NiA1MC41MTQyIDI4LjEwNjZINDMuMDc0NkM0MC43ODk4IDI4LjEwNjYgMzkuMDAyIDI4Ljc2NzMgMzgu"
    "Mjk2NSAyOS44NzM3QzM3Ljc1MTQgMzAuNzI1NSAzNy42MzkyIDMyLjEwMjUgNDAuMTgwNSAzMy43NTgyTDUyLjA0NTQgNDEuNDk1"
    "M0g2MC4xODI1QzYwLjE4MjUgNDEuNDk1MyA0NS4xMzQ5IDMxLjY4ODYgNDQuNzc0MSAzMS40NDk4SDUxLjU0MDRDNTYuOTUxNyAz"
    "MS40NDk4IDYxLjAzMjMgMjguOTUwNCA2MS4wMzIzIDI0LjY5OTgiIGZpbGw9IiMxNDQ3ODIiLz4KPHBhdGggaWQ9IlZlY3Rvcl80"
    "IiBkPSJNOTguMzAyNyA0MS40ODczTDgzLjkyODUgMjAuMDY3QzgyLjU4MTcgMTguMDYxMSA4MC42MDE1IDE3Ljc2NjYgNzkuNTUx"
    "MyAxNy43NjY2Qzc4LjUwMTEgMTcuNzY2NiA3Ni41MTI5IDE4LjA2OTEgNzUuMTc0MSAyMC4wNjdMNjAuNzk5OSA0MS40ODczSDY2"
    "LjgxMjVDNjYuODEyNSA0MS40ODczIDc3LjUzMSAyNS41MTk2IDc5LjU1MTMgMjIuNTAyOEM4MS41NzE1IDI1LjUxMTcgOTIuMjkw"
    "MSA0MS40ODczIDkyLjI5MDEgNDEuNDg3M0g5OC4zMTA3SDk4LjMwMjdaIiBmaWxsPSIjMTQ0NzgyIi8+CjwvZz4KPGcgaWQ9IkNs"
    "aXAgcGF0aCBncm91cCI+CjxtYXNrIGlkPSJtYXNrMF8xMDFfNjg4IiBzdHlsZT0ibWFzay10eXBlOmx1bWluYW5jZSIgbWFza1Vu"
    "aXRzPSJ1c2VyU3BhY2VPblVzZSIgeD0iMzAiIHk9IjciIHdpZHRoPSIyOSIgaGVpZ2h0PSI0NCI+CjxnIGlkPSJjbGlwcGF0aCI+"
    "CjxwYXRoIGlkPSJWZWN0b3JfNSIgZD0iTTU4LjEwNjEgNy4wMjg2OUM0Mi42NDE2IDcuMDI4NjkgMzAuMDQ3MiAxOS41MTc5IDMw"
    "LjAzMTEgMzQuODcyN0MzMC4wMzExIDQwLjY5MTQgMzEuODEwOSA0Ni4yNjM0IDM1LjE4NiA1MC45OTE2TDM3LjgwNzUgNDkuMTQ0"
    "OUMzNC44MTcyIDQ0Ljk1OCAzMy4yNDU5IDQwLjAyMjggMzMuMjQ1OSAzNC44NzI3QzMzLjI2MTkgMjEuMjYxMSA0NC40MjE0IDEw"
    "LjIwNDcgNTguMTIyMiAxMC4yMjA2VjcuMDI4NjlINTguMDk4MSIgZmlsbD0id2hpdGUiLz4KPC9nPgo8L21hc2s+CjxnIG1hc2s9"
    "InVybCgjbWFzazBfMTAxXzY4OCkiPgo8ZyBpZD0iR3JvdXBfMiI+CjxwYXRoIGlkPSJWZWN0b3JfNiIgZD0iTTU2LjIxNDQgMC4w"
    "MDM0OTk3OEwxOC44NzIyIDEwLjAyMTZMMzEuOTM1NSA1OC4wMjYxTDY5LjI3NzcgNDguMDA4TDU2LjIxNDQgMC4wMDM0OTk3OFoi"
    "IGZpbGw9InVybCgjcGFpbnQwX2xpbmVhcl8xMDFfNjg4KSIvPgo8L2c+CjwvZz4KPC9nPgo8ZyBpZD0iR3JvdXBfMyI+CjxwYXRo"
    "IGlkPSJWZWN0b3JfNyIgZD0iTTYxLjkxNDIgNTUuNDI1M1Y1My44ODExSDU4LjM0NjZWNTEuNzAwMUg1NS42NjFWNTMuODgxMUg1"
    "Mi4wOTM1VjU1LjQyNTNINTQuMzk0M0w1NC4yNTgxIDU1LjUyODhDNTMuODAxMSA1NS44OTUgNTMuNDI0MyA1Ni4zNDg3IDUzLjEz"
    "NTcgNTYuODlDNTIuODQ3MSA1Ny40MjMzIDUyLjcwMjggNTguMDEyMyA1Mi43MDI4IDU4LjYzMzJDNTIuNzAyOCA1OS43NDc2IDUz"
    "LjEyNzcgNjAuNzE4NyA1My45Njk0IDYxLjUxNDdDNTQuODExMiA2Mi4zMTg3IDU1LjgzNzQgNjIuNzI0NiA1Ny4wMDc4IDYyLjcy"
    "NDZDNTguMTc4MyA2Mi43MjQ2IDU5LjIwNDUgNjIuMzE4NyA2MC4wMzgyIDYxLjUxNDdDNjAuODcyIDYwLjcxMDggNjEuMjk2OCA1"
    "OS43Mzk2IDYxLjI5NjggNTguNjI1M0M2MS4yOTY4IDU4LjAwNDQgNjEuMTUyNSA1Ny40MjMzIDYwLjg2MzkgNTYuODgyQzYwLjU3"
    "NTMgNTYuMzQwNyA2MC4xOTg1IDU1Ljg4NyA1OS43NDE2IDU1LjUyMDlMNTkuNjEzMyA1NS40MTc0SDYxLjkxNDJWNTUuNDI1M1pN"
    "NTguMjI2NCA2MC40ODc5QzU3Ljg3MzcgNjEuMDIxMiA1Ny40NjQ4IDYxLjI4MzkgNTcuMDA3OCA2MS4yODM5QzU2LjU1MDkgNjEu"
    "MjgzOSA1Ni4xNDIgNjEuMDEzMiA1NS43ODEzIDYwLjQ4NzlDNTUuNDI4NSA1OS45NjI1IDU1LjI1MjEgNTkuMzQxNyA1NS4yNTIx"
    "IDU4LjYzMzJDNTUuMjUyMSA1Ny45MjQ4IDU1LjQyODUgNTcuMzAzOSA1NS43ODEzIDU2Ljc4NjVDNTYuMTM0IDU2LjI2OTEgNTYu"
    "NTUwOSA1NS45OTg1IDU3LjAwNzggNTUuOTk4NUM1Ny40NjQ4IDU1Ljk5ODUgNTcuODczNyA1Ni4yNjExIDU4LjIyNjQgNTYuNzg2"
    "NUM1OC41NzExIDU3LjMwMzkgNTguNzM5NSA1Ny45MjQ4IDU4LjczOTUgNTguNjMzMkM1OC43Mzk1IDU5LjM0MTcgNTguNTYzMSA1"
    "OS45NjI1IDU4LjIyNjQgNjAuNDg3OVoiIGZpbGw9IiM1MTUxNTEiLz4KPHBhdGggaWQ9IlZlY3Rvcl84IiBkPSJNNjYuODA0NCA3"
    "MFY2OC40MzE5SDU3LjkxMzhWNjQuMjIxMUg1NS4yMjAxVjY4LjA4MTdDNTUuMjIwMSA2OC42NzA3IDU1LjQwNDUgNjkuMTQwMyA1"
    "NS43NTcyIDY5LjQ4MjZDNTYuMTE4IDY5LjgyNDkgNTYuNTc0OSA3MCA1Ny4xMTIxIDcwSDY2LjgwNDRaIiBmaWxsPSIjNTE1MTUx"
    "Ii8+CjxwYXRoIGlkPSJWZWN0b3JfOSIgZD0iTTY2LjE3OTEgNTEuOTA3SDYzLjQ5MzVWNjUuMjQ3OUg2Ni4xNzkxVjU3LjY2Mkg2"
    "OC45OTNWNTYuMTAxOUg2Ni4xNzkxVjUxLjkwN1oiIGZpbGw9IiM1MTUxNTEiLz4KPHBhdGggaWQ9IlZlY3Rvcl8xMCIgZD0iTTg1"
    "LjM4NzUgNjAuNDcyVjU4LjkwMzlIODIuNjQ1OEw4Mi43MDk5IDU4LjgxNjNDODIuODg2MyA1OC41NjE2IDgzLjA0NjYgNTguMTM5"
    "NyA4My4xODI5IDU3LjU2NjZDODMuMzExMiA1Ni45OTM1IDgzLjM3NTMgNTYuNDQ0MiA4My4zNzUzIDU1LjkzNDhWNTEuOTk0Nkg3"
    "MS41MTg0VjUzLjU2MjdIODAuNjg5NlY1Ni4wMTQ0QzgwLjY4OTYgNTYuNTIzOSA4MC42NjU2IDU3LjAzMzMgODAuNjI1NSA1Ny41"
    "MzQ4QzgwLjU4NTQgNTguMDM2MiA4MC40ODkyIDU4LjQ4MiA4MC4zNDQ5IDU4Ljg2NDFMODAuMzI4OSA1OC45MDM5SDY5LjU4NjNW"
    "NjAuNDcySDc2LjA4OFY2My41MDQ3SDcxLjQ0NjJWNjUuMDY0OUg4MC42NTc2VjY5Ljk5MjFIODMuMzQzMlY2My40OTY4SDc4Ljc4"
    "MTZWNjAuNDY0SDg1LjM5NTVMODUuMzg3NSA2MC40NzJaIiBmaWxsPSIjNTE1MTUxIi8+CjxwYXRoIGlkPSJWZWN0b3JfMTEiIGQ9"
    "Ik05Ni4wOTggNTIuNDk2MUg4Ny41ODQxVjYxLjU2MjVDODcuNTg0MSA2Mi4xMzU2IDg3Ljc2MDUgNjIuNjA1MyA4OC4wOTcyIDYy"
    "Ljk0NzVDODguNDMzOSA2My4yODk4IDg4Ljg5MDkgNjMuNDY0OSA4OS40NiA2My40NjQ5SDk2LjA5OFY1Mi40OTYxWk05MC4yNjk4"
    "IDU0LjA1NjNIOTMuNDA0M1Y2MS44OTY4SDkwLjI2OThWNTQuMDU2M1oiIGZpbGw9IiM1MTUxNTEiLz4KPHBhdGggaWQ9IlZlY3Rv"
    "cl8xMiIgZD0iTTEwMS4wNiA1MS45MDdIOTguMzY2OFY2OS45OTJIMTAxLjA2VjU3LjkyNDdIMTAzLjg3NFY1Ni4zNTY2SDEwMS4w"
    "NlY1MS45MDdaIiBmaWxsPSIjNTE1MTUxIi8+CjxwYXRoIGlkPSJWZWN0b3JfMTMiIGQ9Ik0xMTQuOTA2IDYyLjQzMDFDMTEzLjgw"
    "NyA2Mi4xODM0IDExMi44MTMgNjEuNDUxMSAxMTEuOTU1IDYwLjI0OTFDMTExLjA3MyA1OS4wMDc0IDExMC42MTcgNTcuNTAyOSAx"
    "MTAuNjE3IDU1Ljc1MTdWNTIuMzA1MUgxMDcuODkxVjU1Ljc1MTdDMTA3Ljg5MSA1Ny40ODcgMTA3LjQ1IDU4Ljk5MTQgMTA2LjU3"
    "NiA2MC4yNDExQzEwNS43MjYgNjEuNDUxMSAxMDQuNzMyIDYyLjE4MzQgMTAzLjYxIDYyLjQzMDFMMTA0LjkwMSA2My42MjQxQzEw"
    "NS43ODIgNjMuNTI4NiAxMDYuNjcyIDYzLjExNDcgMTA3LjU2MiA2Mi4zOTgzQzEwOC40NiA2MS42NjYgMTA5LjAxMyA2MC44ODU5"
    "IDEwOS4yMDYgNjAuMDc0TDEwOS4yNjIgNTkuODQzMkwxMDkuMzI2IDYwLjA2NkMxMDkuNTQyIDYwLjg4NTkgMTEwLjEwMyA2MS42"
    "NjYgMTEwLjk5MyA2Mi4zOTgzQzExMS44NjcgNjMuMTIyNyAxMTIuNzY1IDYzLjUzNjYgMTEzLjY1NSA2My42MTYyTDExNC45MTQg"
    "NjIuNDIyMkwxMTQuOTA2IDYyLjQzMDFaIiBmaWxsPSIjNTE1MTUxIi8+CjxwYXRoIGlkPSJWZWN0b3JfMTQiIGQ9Ik0xMTguNDcz"
    "IDUxLjkwN0gxMTUuNzc5VjY5Ljk5MkgxMTguNDczVjU3LjgyOTJIMTIxLjI3OVY1Ni4yNjExSDExOC40NzNWNTEuOTA3WiIgZmls"
    "bD0iIzUxNTE1MSIvPgo8cGF0aCBpZD0iVmVjdG9yXzE1IiBkPSJNMTM0LjMwNiA1MS45MDdWNjMuMzUzNUwxMzQuMjY2IDYzLjM2"
    "OTRDMTMzLjkyMiA2My40ODg4IDEzMy40ODkgNjMuNTg0MyAxMzIuOTc2IDYzLjY3MTlDMTMyLjQ3MSA2My43NTE1IDEzMS45MDkg"
    "NjMuNzkxMyAxMzEuMjkyIDYzLjc5MTNIMTI5LjA3OVY2Mi4zNjY0TDEyOS4xMTkgNjIuMzUwNUMxMjkuOTUzIDYyLjA3MTkgMTMw"
    "LjY1MSA2MS41NjI1IDEzMS4xOCA2MC44NjJDMTMxLjcxNyA2MC4xNTM2IDEzMS45ODEgNTkuMzQ5NiAxMzEuOTgxIDU4LjQ3NEMx"
    "MzEuOTgxIDU3LjkzMjcgMTMxLjg2OSA1Ny40MDc0IDEzMS42NDUgNTYuOTIxOEMxMzEuNDI4IDU2LjQzNjMgMTMxLjEyNCA1Ni4w"
    "MDY0IDEzMC43NjMgNTUuNjQwM0wxMzAuNjU5IDU1LjUzNjhIMTMyLjY5NVY1My45OTI2SDEyOS4wNzFWNTEuODExNUgxMjYuMzc4"
    "VjUzLjk5MjZIMTIyLjc3VjU1LjUzNjhIMTI0LjgxNEwxMjQuNzEgNTUuNjQwM0MxMjQuMzMzIDU1Ljk5ODUgMTI0LjAzNyA1Ni40"
    "MjgzIDEyMy44MTIgNTYuOTIxOEMxMjMuNTk2IDU3LjQwNzQgMTIzLjQ5MiA1Ny45MzI3IDEyMy40OTIgNTguNDc0QzEyMy40OTIg"
    "NTkuMzU3NiAxMjMuNzY0IDYwLjE2MTUgMTI0LjI5MyA2MC44NjJDMTI0LjgyMiA2MS41NzA1IDEyNS41MTIgNjIuMDY0IDEyNi4z"
    "MzggNjIuMzM0NkgxMjYuMzc4VjYzLjc5MTNIMTIyLjY2NlY2NS4zNTk0SDEzMS4xOEMxMzEuNzU3IDY1LjM1OTQgMTMyLjMxOCA2"
    "NS4yOTU3IDEzMi44MzkgNjUuMTY4NEMxMzMuMzYgNjUuMDQxIDEzMy44MjUgNjQuODk3NyAxMzQuMjAyIDY0LjcyMjZMMTM0LjI4"
    "MiA2NC42ODI4VjcwLjAwMDFIMTM2Ljk3NlY1MS45MDdIMTM0LjI4MkgxMzQuMzA2Wk0xMjcuNzQ5IDYxLjA5MjlDMTI3LjI5MiA2"
    "MS4wOTI5IDEyNi44ODMgNjAuODIyMiAxMjYuNTU0IDYwLjI5NjlDMTI2LjIyNSA1OS43Nzk1IDEyNi4wNTcgNTkuMTU4NiAxMjYu"
    "MDU3IDU4LjQ2NjFDMTI2LjA1NyA1Ny43NzM1IDEyNi4yMjUgNTcuMTUyNyAxMjYuNTU0IDU2LjYzNTNDMTI2Ljg5MSA1Ni4xMDIg"
    "MTI3LjI5MiA1NS44MzkzIDEyNy43NDkgNTUuODM5M0MxMjguMjA2IDU1LjgzOTMgMTI4LjYyMiA1Ni4xMDk5IDEyOC45NTkgNTYu"
    "NjM1M0MxMjkuMjggNTcuMTUyNyAxMjkuNDQ4IDU3Ljc3MzUgMTI5LjQ0OCA1OC40NjYxQzEyOS40NDggNTkuMTU4NiAxMjkuMjg4"
    "IDU5Ljc3OTUgMTI4Ljk1OSA2MC4yOTY5QzEyOC42MyA2MC44MzAyIDEyOC4yMjIgNjEuMDkyOSAxMjcuNzQ5IDYxLjA5MjlaIiBm"
    "aWxsPSIjNTE1MTUxIi8+CjwvZz4KPC9nPgo8ZGVmcz4KPGxpbmVhckdyYWRpZW50IGlkPSJwYWludDBfbGluZWFyXzEwMV82ODgi"
    "IHgxPSI0Ny41OTQ1IiB5MT0iNDYuNTEyNyIgeDI9IjQwLjUxMDgiIHkyPSIyMC4xMDg0IiBncmFkaWVudFVuaXRzPSJ1c2VyU3Bh"
    "Y2VPblVzZSI+CjxzdG9wIHN0b3AtY29sb3I9IndoaXRlIi8+CjxzdG9wIG9mZnNldD0iMC4wMiIgc3RvcC1jb2xvcj0iI0VERjhG"
    "OCIvPgo8c3RvcCBvZmZzZXQ9IjAuMDkiIHN0b3AtY29sb3I9IiNCQ0U3RTMiLz4KPHN0b3Agb2Zmc2V0PSIwLjE3IiBzdG9wLWNv"
    "bG9yPSIjOEZEN0QxIi8+CjxzdG9wIG9mZnNldD0iMC4yNSIgc3RvcC1jb2xvcj0iIzY4Q0FDMSIvPgo8c3RvcCBvZmZzZXQ9IjAu"
    "MzQiIHN0b3AtY29sb3I9IiM0OEJFQjQiLz4KPHN0b3Agb2Zmc2V0PSIwLjQzIiBzdG9wLWNvbG9yPSIjMkRCNUE5Ii8+CjxzdG9w"
    "IG9mZnNldD0iMC41MyIgc3RvcC1jb2xvcj0iIzE5QURBMSIvPgo8c3RvcCBvZmZzZXQ9IjAuNjQiIHN0b3AtY29sb3I9IiMwQkE4"
    "OUIiLz4KPHN0b3Agb2Zmc2V0PSIwLjc4IiBzdG9wLWNvbG9yPSIjMDJBNTk4Ii8+CjxzdG9wIG9mZnNldD0iMSIgc3RvcC1jb2xv"
    "cj0iIzAwQTU5NyIvPgo8L2xpbmVhckdyYWRpZW50Pgo8Y2xpcFBhdGggaWQ9ImNsaXAwXzEwMV82ODgiPgo8cmVjdCB3aWR0aD0i"
    "MTM3IiBoZWlnaHQ9IjcwIiBmaWxsPSJ3aGl0ZSIvPgo8L2NsaXBQYXRoPgo8L2RlZnM+Cjwvc3ZnPgo="
)

def get_logo_data_uri():
    """리포트에 삽입할 로고를 data URI로 반환."""
    if LOGO_OVERRIDE_PATH and os.path.exists(LOGO_OVERRIDE_PATH):
        ext = os.path.splitext(LOGO_OVERRIDE_PATH)[1].lower()
        mime = {
            ".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
            ".svg": "image/svg+xml", ".gif": "image/gif", ".webp": "image/webp",
        }.get(ext, "image/png")
        with open(LOGO_OVERRIDE_PATH, "rb") as f:
            return f"data:{mime};base64," + base64.b64encode(f.read()).decode("ascii")

    return "data:image/svg+xml;base64," + _KRA_LOGO_B64


# ======================================================================
# 데이터 로드
# ======================================================================
def load_intro(path):
    """자기소개서 원본: 면접번호 -> {모집분야, 전형단계, qa:[{question, answer}]}"""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["자기소개서"]

    data = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[0] is None:
            continue
        exam_no, name, field, status, question, answer = row[:6]
        if exam_no not in data:
            data[exam_no] = {"모집분야": field, "전형단계": status, "qa": []}
        data[exam_no]["qa"].append({
            "question": question or "",
            "answer": answer or "",
        })
    return data


def load_interview_questions(path):
    """면접질문 결과: 면접번호 -> {모집분야, main_questions:[...]}
    헤더를 읽어 '근거구절' 컬럼 유무를 자동 판단(구버전 파일 호환)."""
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["면접질문"]

    rows = list(ws.iter_rows(min_row=1, values_only=True))
    if not rows:
        return {}

    header = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx = {name: i for i, name in enumerate(header)}

    def get(row, key, default=None):
        idx = col_idx.get(key)
        if idx is None or idx >= len(row):
            return default
        return row[idx]

    data = {}
    for row in rows[1:]:
        if row[0] is None:
            continue

        exam_no = get(row, "면접번호")
        field = get(row, "모집분야")
        factor = get(row, "평가요소", "")
        question = get(row, "주질문", "")
        source_quote = get(row, "근거구절", "")
        reason = get(row, "질문이유", "")
        sub_text = get(row, "보조질문", "")

        if exam_no not in data:
            data[exam_no] = {"모집분야": field, "main_questions": []}

        sub_list = []
        if sub_text:
            sub_list = [s.strip() for s in str(sub_text).split("\n") if s.strip()]

        data[exam_no]["main_questions"].append({
            "factor": factor or "",
            "question": question or "",
            "source_quote": source_quote or "",
            "reason": reason or "",
            "sub_questions": sub_list,
        })
    return data


def load_summaries(path):
    """문항요약 시트: 면접번호 -> {문항번호: {headline, points}}
    시트가 없으면 빈 dict 반환(요약 없이도 리포트는 정상 생성)."""
    wb = openpyxl.load_workbook(path, data_only=True)
    if "문항요약" not in wb.sheetnames:
        return {}

    ws = wb["문항요약"]
    rows = list(ws.iter_rows(min_row=1, values_only=True))
    if not rows:
        return {}

    header = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx = {name: i for i, name in enumerate(header)}

    def get(row, key, default=""):
        idx = col_idx.get(key)
        if idx is None or idx >= len(row):
            return default
        return row[idx] if row[idx] is not None else default

    data = {}
    for row in rows[1:]:
        if row[0] is None:
            continue
        exam_no = get(row, "면접번호")
        try:
            q_no = int(get(row, "문항번호", 0))
        except (TypeError, ValueError):
            continue

        points = [p.strip() for p in str(get(row, "요약항목", "")).split("\n") if p.strip()]

        def _int(key):
            try:
                return int(get(row, key, 0) or 0)
            except (TypeError, ValueError):
                return 0

        data.setdefault(exam_no, {})[q_no] = {
            "headline": str(get(row, "핵심요약", "")).strip(),
            "points": points,
            "chars": _int("작성글자수"),
            "chars_nospace": _int("공백제외글자수"),
        }
    return data


# ======================================================================
# 요약이 없을 때 사용하는 자동 추출 요약(대체 수단)
# ======================================================================
def fallback_summary(answer, max_points=2, max_len=60):
    text = re.sub(r"\s+", " ", str(answer or "")).strip()
    if not text:
        return {"headline": "미작성", "points": []}

    sentences = [s.strip() for s in re.split(r"(?<=[.!?다])\s+", text) if s.strip()]
    points = []
    for s in sentences[:max_points]:
        points.append(s if len(s) <= max_len else s[:max_len].rstrip() + "…")

    head = sentences[0] if sentences else text
    head = head if len(head) <= 40 else head[:40].rstrip() + "…"
    return {"headline": head, "points": points}


# ======================================================================
# 근거구절 -> 자소서 문항 매칭
# ======================================================================
def _normalize(text):
    return re.sub(r"\s+", "", str(text or ""))


def find_source_question_no(source_quote, qa_list, min_len=8):
    quote_norm = _normalize(source_quote)
    if len(quote_norm) < min_len:
        return None

    for i, qa in enumerate(qa_list, start=1):
        answer_norm = _normalize(qa.get("answer", ""))
        if quote_norm and quote_norm in answer_norm:
            return i
        head = quote_norm[:min(len(quote_norm), 20)]
        if len(head) >= min_len and head in answer_norm:
            return i
    return None


# ======================================================================
# 공통 스타일
# ======================================================================
COMMON_STYLE = """
  * { box-sizing: border-box; }
  body {
    font-family: "Malgun Gothic", "Apple SD Gothic Neo", sans-serif;
    color: #222;
    line-height: 1.6;
  }
  .container {
    max-width: 980px;
    margin: 0 auto;
    background: #fff;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    overflow: hidden;
  }
  .header {
    background: #144782;
    color: #fff;
    padding: 24px 32px;
  }
  .header-top {
    display: flex;
    align-items: center;
    gap: 16px;
    margin-bottom: 10px;
  }
  .logo-img {
    height: 42px;
    width: auto;
    flex-shrink: 0;
    background: #fff;
    border-radius: 6px;
    padding: 5px 10px;
  }
  .header h1 {
    margin: 0;
    font-size: 21px;
    line-height: 1.35;
  }
  .header .meta { font-size: 14px; color: #cfe0f5; }
  .badge {
    display: inline-block;
    background: rgba(255,255,255,0.18);
    border-radius: 4px;
    padding: 2px 8px;
    font-size: 12px;
    margin-right: 6px;
  }
  .section {
    padding: 24px 32px;
    border-bottom: 1px solid #eee;
  }
  .section h2 {
    font-size: 18px;
    color: #144782;
    border-left: 5px solid #00A597;
    padding-left: 10px;
    margin-top: 0;
  }
  .section h2 .h2-sub {
    display: block;
    font-size: 12px;
    font-weight: normal;
    color: #7a8b9c;
    margin-top: 3px;
  }

  /* ---------- 리포트 안내 콜아웃 ---------- */
  .notice {
    margin: 22px 32px 4px 32px;
    background: #f1f8f7;
    border: 1px solid #bfe0dc;
    border-left: 5px solid #00A597;
    border-radius: 8px;
    padding: 15px 20px;
  }
  .notice-title {
    font-size: 13px;
    font-weight: bold;
    color: #00695c;
    letter-spacing: 0.5px;
    margin-bottom: 6px;
  }
  .notice-body {
    margin: 0;
    font-size: 13.5px;
    color: #2f4f4a;
    line-height: 1.75;
  }

  /* ---------- 1. 문항별 요약 ---------- */
  .summary-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 14px;
    table-layout: fixed;
  }
  .summary-table th {
    background: #144782;
    color: #fff;
    font-size: 13px;
    font-weight: bold;
    text-align: center;
    padding: 10px 12px;
    border: 1px solid #144782;
  }
  .summary-table th.col-no { width: 78px; }
  .summary-table th.col-q  { width: 34%; }
  .summary-table td {
    border: 1px solid #d9e2ec;
    padding: 12px;
    vertical-align: top;
  }
  .summary-table tr:nth-child(even) td { background: #fafbfc; }
  .summary-table td.no-cell {
    text-align: center;
    font-weight: bold;
    color: #144782;
    background: #eaf1fb !important;
    white-space: nowrap;
  }
  .summary-table td.q-cell {
    color: #33475b;
    font-size: 13px;
  }
  .summary-headline {
    font-weight: bold;
    color: #0f2f57;
    margin-bottom: 6px;
  }
  .summary-points {
    margin: 0;
    padding-left: 16px;
    color: #444;
    font-size: 13.5px;
  }
  .summary-points li { margin-bottom: 3px; }
  .summary-empty { color: #9aa7b4; font-style: italic; }
  .char-badge {
    display: inline-block;
    background: #eef4fb;
    border: 1px solid #d3e0ef;
    border-radius: 12px;
    padding: 1px 10px;
    font-size: 12px;
    color: #144782;
    font-weight: bold;
    margin-bottom: 7px;
  }
  .char-badge .char-sub {
    font-weight: normal;
    color: #7a8b9c;
    margin-left: 4px;
  }
  .char-badge.is-empty {
    background: #f6f7f8;
    border-color: #e0e4e8;
    color: #9aa7b4;
  }
  .summary-note {
    margin-top: 10px;
    font-size: 12px;
    color: #9aa7b4;
  }

  /* ---------- 자기소개서 원본 ---------- */
  .intro-item {
    background: #fafbfc;
    border: 1px solid #e3e7eb;
    border-radius: 8px;
    padding: 16px 18px;
    margin-bottom: 14px;
  }
  .intro-item .intro-no {
    display: inline-block;
    background: #607d8b;
    color: #fff;
    font-size: 11px;
    font-weight: bold;
    border-radius: 4px;
    padding: 2px 6px;
    margin-right: 6px;
  }
  .intro-q { font-weight: bold; color: #144782; margin-bottom: 8px; }
  .intro-a { white-space: pre-wrap; color: #444; font-size: 14px; }

  /* ---------- 면접질문 ---------- */
  .main-q-block {
    border: 1px solid #d9e2ec;
    border-radius: 10px;
    margin-bottom: 18px;
    overflow: hidden;
  }
  .main-q-header { background: #eaf1fb; padding: 14px 18px; }
  .main-q-no {
    display: inline-block;
    background: #144782;
    color: #fff;
    font-size: 12px;
    font-weight: bold;
    border-radius: 4px;
    padding: 2px 8px;
    margin-right: 8px;
  }
  .main-q-text { font-weight: bold; font-size: 15px; color: #1a1a1a; }
  .source-box {
    margin-top: 10px;
    font-size: 13px;
    color: #5d4037;
    background: #fff3e0;
    border-left: 3px solid #ff9800;
    border-radius: 6px;
    padding: 8px 12px;
  }
  .source-box .label { font-weight: bold; margin-right: 4px; color: #e65100; }
  .source-box .quote { font-style: italic; }
  .source-box .source-ref {
    display: block; margin-top: 4px; font-size: 12px; color: #9e7b52;
  }
  .reason-box {
    margin-top: 10px;
    font-size: 13px;
    color: #00695c;
    background: #e6f4f1;
    border-radius: 6px;
    padding: 8px 12px;
  }
  .reason-box .label { font-weight: bold; margin-right: 4px; }
  .sub-q-list { padding: 14px 18px 16px 18px; }
  .sub-q-list .sub-title {
    font-size: 13px;
    font-weight: bold;
    color: #888;
    margin-bottom: 8px;
    letter-spacing: 0.5px;
  }
  .sub-q-item {
    padding: 8px 0 8px 28px;
    position: relative;
    font-size: 14px;
    color: #333;
    border-top: 1px dashed #e0e0e0;
  }
  .sub-q-item:first-child { border-top: none; }
  .sub-q-item::before {
    content: "▸";
    position: absolute;
    left: 6px;
    color: #00A597;
    font-weight: bold;
  }

  /* ---------- 발표력 행동지표 표 ---------- */
  .behavior-wrap { padding: 16px 18px 18px 18px; }
  .behavior-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13.5px;
    text-align: left;
  }
  .behavior-table th {
    background: #eaf1fb;
    border: 1px solid #d9e2ec;
    padding: 9px 10px;
    color: #144782;
  }
  .behavior-table td { border: 1px solid #d9e2ec; padding: 9px 10px; }
  .behavior-table td.item {
    font-weight: bold;
    color: #144782;
    background: #fafbfc;
    white-space: nowrap;
  }

  .footer {
    padding: 16px 32px;
    font-size: 12px;
    color: #999;
    text-align: center;
  }

  /* ---------- 인쇄(PDF) ---------- */
  /* 화면과 동일한 디자인 그대로 인쇄되도록 색상을 강제한다.
     print-color-adjust: exact 를 쓰면 크롬의 "배경 그래픽" 체크 여부와
     무관하게 배경색·배지 색상이 그대로 출력된다. */
  @media print {
    *, *::before, *::after {
      -webkit-print-color-adjust: exact !important;
      print-color-adjust: exact !important;
    }

    body {
      background: #fff !important;
      margin: 0 !important;
      padding: 0 !important;
    }
    .container {
      max-width: none;
      box-shadow: none;
      border-radius: 0;
    }

    /* --- 제목이 본문과 떨어져 홀로 남지 않게 --- */
    .section h2 {
      page-break-inside: avoid;
      break-inside: avoid;
      page-break-after: avoid;
      break-after: avoid;
    }
    .header {
      page-break-inside: avoid;
      break-inside: avoid;
      page-break-after: avoid;
      break-after: avoid;
    }

    /* --- 블록이 페이지 경계에서 잘리지 않게 --- */
    .notice,
    .main-q-block,
    .intro-item,
    .summary-table tr,
    .behavior-table tr {
      page-break-inside: avoid;
      break-inside: avoid;
    }

    /* 발표력 행동지표 블록은 라벨/제목과 표가 갈라지지 않도록 통째로 이동 */
    .presentation-block {
      page-break-inside: avoid;
      break-inside: avoid;
    }
    .presentation-block .main-q-header {
      page-break-after: avoid;
      break-after: avoid;
    }
    .behavior-wrap {
      page-break-before: avoid;
      break-before: avoid;
    }

    /* 질문 블록 내부에서도 주질문과 보조질문이 갈라지지 않게 */
    .main-q-header {
      page-break-after: avoid;
      break-after: avoid;
    }
    .sub-q-list {
      page-break-before: avoid;
      break-before: avoid;
    }

    /* 표가 여러 페이지에 걸치면 제목행 반복 */
    .summary-table thead,
    .behavior-table thead { display: table-header-group; }

    p, li { orphans: 3; widows: 3; }
    a { text-decoration: none; color: inherit; }
  }
"""


# ======================================================================
# 렌더링 - 1. 자기소개서 문항별 요약
# ======================================================================
def count_chars(text):
    """작성 글자수(공백 포함, 공백 제외)"""
    t = str(text or "")
    return len(t), len(re.sub(r"\s", "", t))


def render_char_badge(answer):
    total, nospace = count_chars(answer)
    if total == 0:
        return '<div class="char-badge is-empty">미작성</div>'
    return (
        f'<div class="char-badge">{total:,}자'
        f'<span class="char-sub">공백 제외 {nospace:,}자</span></div>'
    )


def render_summary_html(qa_list, summary_map):
    """summary_map: {문항번호: {headline, points}} / 없으면 자동 추출 요약 사용"""
    if not qa_list:
        return "<p>자기소개서 데이터가 없습니다.</p>"

    used_fallback = False
    rows = []

    for i, qa in enumerate(qa_list, start=1):
        s = (summary_map or {}).get(i)
        if not s or (not s.get("headline") and not s.get("points")):
            s = fallback_summary(qa.get("answer", ""))
            used_fallback = True

        q_text = html.escape(str(qa.get("question", "")))
        headline = html.escape(str(s.get("headline", "")))
        points = s.get("points", []) or []
        char_html = render_char_badge(qa.get("answer", ""))

        if points:
            points_html = (
                '<ul class="summary-points">'
                + "".join(f"<li>{html.escape(str(p))}</li>" for p in points)
                + "</ul>"
            )
        else:
            points_html = ""

        if count_chars(qa.get("answer", ""))[0] == 0:
            body_html = char_html
        elif not headline and not points_html:
            body_html = char_html + '<span class="summary-empty">요약 정보 없음</span>'
        else:
            body_html = (
                char_html
                + (f'<div class="summary-headline">{headline}</div>' if headline else "")
                + points_html
            )

        rows.append(f"""
          <tr>
            <td class="no-cell">문항 {i}</td>
            <td class="q-cell">{q_text}</td>
            <td>{body_html}</td>
          </tr>""")

    note = ""
    if used_fallback:
        note = ('<div class="summary-note">※ 일부 문항은 자기소개서 원문에서 '
                '자동 추출한 내용입니다.</div>')

    return f"""
        <table class="summary-table">
          <thead>
            <tr>
              <th class="col-no">구분</th>
              <th class="col-q">자기소개서 문항</th>
              <th>작성 내용 요약</th>
            </tr>
          </thead>
          <tbody>{"".join(rows)}
          </tbody>
        </table>
        {note}"""


# ======================================================================
# 렌더링 - 자기소개서 원본
# ======================================================================
def render_intro_html(qa_list):
    parts = []
    for i, qa in enumerate(qa_list, start=1):
        q = html.escape(str(qa["question"]))
        a = html.escape(str(qa["answer"]))
        parts.append(f"""
        <div class="intro-item">
          <div class="intro-q"><span class="intro-no">문항 {i}</span>{q}</div>
          <div class="intro-a">{a}</div>
        </div>""")
    return "\n".join(parts) if parts else "<p>자기소개서 데이터가 없습니다.</p>"


# ======================================================================
# 렌더링 - 면접질문
# ======================================================================
# GPT가 생성하는 주질문의 평가요소 순서 (발표력은 아래 고정표로 대체)
EVAL_FACTOR_LABELS = ["정신자세", "전문지식", "품행", "발전가능성"]

# 발표력 고정 블록을 삽입할 위치 (0-based: 2 → 전문지식 다음)
PRESENTATION_INSERT_AT = 2

PRESENTATION_ROWS = [
    ("핵심전달력", "핵심 내용을 명확하고 간결하게 전달함", "핵심이 불분명하거나 장황하게 설명함"),
    ("논리성", "내용 전개가 체계적이고 논리적임", "내용 간 연결성이 부족하고 논리 전개가 미흡함"),
    ("표현력", "적절한 속도·발성·억양으로 이해하기 쉽게 설명함", "말이 너무 빠르거나 느리고 발음이 부정확함"),
    ("질의응답", "질문 의도를 정확히 파악하고 적절히 답변함", "질문 의도를 잘못 이해하거나 답변이 엉뚱함"),
    ("태도", "자신감 있고 안정적인 자세를 유지함", "지나치게 위축되거나 불안한 모습을 보임"),
    ("청중소통", "면접관과 시선 교환을 하며 자연스럽게 소통함", "시선 회피, 원고만 읽기 등 소통이 부족함"),
]


def render_presentation_block():
    body_rows = "".join(
        f"<tr><td class=\"item\">{html.escape(item)}</td>"
        f"<td>{html.escape(pos)}</td><td>{html.escape(neg)}</td></tr>"
        for item, pos, neg in PRESENTATION_ROWS
    )
    return f"""
        <div class="main-q-block presentation-block">
          <div class="main-q-header">
            <div><span class="main-q-no">발표력</span></div>
            <div class="main-q-text">발표력 평가 행동지표 (면접관 관찰용)</div>
          </div>
          <div class="behavior-wrap">
            <table class="behavior-table">
              <thead>
                <tr><th style="width:16%">평가항목</th><th>긍정요소</th><th>부정요소</th></tr>
              </thead>
              <tbody>{body_rows}</tbody>
            </table>
          </div>
        </div>"""


def get_main_q_label(index_1based, mq=None):
    """엑셀에 '평가요소' 컬럼이 있으면 그 값을, 없으면 순서 기준 라벨을 사용."""
    if mq:
        factor = str(mq.get("factor") or "").strip()
        if factor:
            return factor
    if 1 <= index_1based <= len(EVAL_FACTOR_LABELS):
        return EVAL_FACTOR_LABELS[index_1based - 1]
    return f"주질문 {index_1based}"


def render_one_question(i, mq, qa_list):
    q = html.escape(str(mq["question"]))
    reason = html.escape(str(mq.get("reason", "")))
    source_quote = str(mq.get("source_quote", "")).strip()

    sub_items = "".join(
        f'<div class="sub-q-item">{html.escape(str(s))}</div>'
        for s in mq.get("sub_questions", [])
    )

    reason_html = (
        f'<div class="reason-box"><span class="label">질문이유</span>{reason}</div>'
        if reason else ""
    )

    source_html = ""
    if source_quote:
        quote_escaped = html.escape(source_quote)
        q_no = find_source_question_no(source_quote, qa_list)
        ref_text = (f"출처: 자기소개서 문항 {q_no}" if q_no
                    else "출처: 자기소개서 (정확한 문항 매칭 안됨)")
        source_html = (
            f'<div class="source-box">'
            f'<span class="label">근거구절</span>'
            f'<span class="quote">“{quote_escaped}”</span>'
            f'<span class="source-ref">{ref_text}</span>'
            f'</div>'
        )

    return f"""
        <div class="main-q-block">
          <div class="main-q-header">
            <div><span class="main-q-no">{html.escape(get_main_q_label(i, mq))}</span></div>
            <div class="main-q-text">{q}</div>
            {source_html}
            {reason_html}
          </div>
          <div class="sub-q-list">
            <div class="sub-title">보조 질문</div>
            {sub_items if sub_items else '<div class="sub-q-item">(보조질문 없음)</div>'}
          </div>
        </div>"""


def render_questions_html(main_questions, qa_list):
    if not main_questions:
        return render_presentation_block()

    parts = []
    for i, mq in enumerate(main_questions, start=1):
        if i - 1 == PRESENTATION_INSERT_AT:
            parts.append(render_presentation_block())
        parts.append(render_one_question(i, mq, qa_list))

    if len(main_questions) <= PRESENTATION_INSERT_AT:
        parts.append(render_presentation_block())

    return "\n".join(parts)


# ======================================================================
# 리포트 본문
# ======================================================================
def render_report_body(exam_no, field, intro_qa, main_questions, summary_map):
    logo_uri = get_logo_data_uri()
    logo_html = (f'<img src="{logo_uri}" alt="한국마사회" class="logo-img">'
                 if logo_uri else "")

    notice_html = html.escape(NOTICE_TEXT).replace("\n", "<br>")

    return f"""
  <div class="header">
    <div class="header-top">
      {logo_html}
      <h1>{html.escape(REPORT_TITLE)}</h1>
    </div>
    <div class="meta">
      <span class="badge">면접번호: {html.escape(str(exam_no))}</span>
      <span class="badge">모집분야: {html.escape(str(field or ""))}</span>
      <span class="badge">이름: ●●●(블라인드)</span>
    </div>
  </div>

  <div class="notice">
    <div class="notice-title">면접위원 안내</div>
    <p class="notice-body">{notice_html}</p>
  </div>

  <div class="section">
    <h2>1. 자기소개서 문항별 내용 요약
      <span class="h2-sub">지원자가 작성한 내용을 사실 그대로 요약한 것으로, 평가 결과가 아닙니다.</span>
    </h2>
    {render_summary_html(intro_qa, summary_map)}
  </div>

  <div class="section">
    <h2>2. 맞춤형 면접 질문
      <span class="h2-sub">정신자세 · 전문지식 · 발표력 · 품행 · 발전가능성</span>
    </h2>
    {render_questions_html(main_questions, intro_qa)}
  </div>

  <div class="section">
    <h2>3. 자기소개서 원본</h2>
    {render_intro_html(intro_qa)}
  </div>

  <div class="footer">
    본 리포트는 생성형 AI 기술을 활용하여 지원자가 작성한 자기소개서 내용을 바탕으로 맞춤 제작되었습니다.
  </div>
"""


# ======================================================================
# 개별 리포트 HTML
# ======================================================================
INDIVIDUAL_TEMPLATE = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>{title} - {exam_no}</title>
<style>
  body {{ background: #f4f6f8; margin: 0; padding: 24px; }}
  @page {{ size: A4; margin: 10mm; }}
{common_style}
</style>
</head>
<body>
<div class="container">
{body}
</div>
</body>
</html>
"""


def build_individual_report(exam_no, field, intro_qa, main_questions, summary_map):
    body = render_report_body(exam_no, field, intro_qa, main_questions, summary_map)
    return INDIVIDUAL_TEMPLATE.format(
        title=html.escape(REPORT_TITLE),
        exam_no=html.escape(str(exam_no)),
        common_style=COMMON_STYLE,
        body=body,
    )


# ======================================================================
# 통합 리포트(사이드바) HTML
# ======================================================================
COMBINED_TEMPLATE = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<title>{title} - 통합 보기</title>
<style>
  @page {{ size: A4; margin: 10mm; }}
  * {{ box-sizing: border-box; }}
  html, body {{
    margin: 0;
    height: 100%;
    font-family: "Malgun Gothic", "Apple SD Gothic Neo", sans-serif;
    background: #f4f6f8;
  }}
  .layout {{ display: flex; height: 100vh; }}
  .sidebar {{
    width: 260px;
    flex-shrink: 0;
    background: #0f2f57;
    color: #fff;
    overflow-y: auto;
    padding: 16px 0;
  }}
  .sidebar-logo {{
    display: block;
    height: 40px;
    width: auto;
    background: #fff;
    border-radius: 6px;
    padding: 5px 10px;
    margin: 0 16px 16px 16px;
  }}
  .sidebar h2 {{
    font-size: 13px;
    color: #9fc1e8;
    padding: 0 16px;
    margin: 0 0 10px 0;
    letter-spacing: 1px;
  }}
  .sidebar .item {{
    display: block;
    width: 100%;
    text-align: left;
    background: none;
    border: none;
    color: #d6e4f7;
    padding: 10px 16px;
    font-size: 14px;
    cursor: pointer;
    border-left: 3px solid transparent;
  }}
  .sidebar .item .field {{
    display: block; font-size: 11px; color: #8fb2dd; margin-top: 2px;
  }}
  .sidebar .item:hover {{ background: #16407a; }}
  .sidebar .item.active {{
    background: #144782;
    border-left-color: #00A597;
    color: #fff;
    font-weight: bold;
  }}
  .content {{ flex: 1; overflow-y: auto; padding: 24px; }}
  .report-panel {{
    display: none;
    max-width: 980px;
    margin: 0 auto;
    background: #fff;
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    overflow: hidden;
  }}
  .report-panel.active {{ display: block; }}
  .placeholder {{
    max-width: 980px; margin: 60px auto; text-align: center;
    color: #999; font-size: 15px;
  }}
{common_style}
  @media print {{
    .sidebar {{ display: none; }}
    .layout {{ display: block; height: auto; }}
    .content {{ overflow: visible; padding: 0; }}
    .placeholder {{ display: none; }}
    .report-panel.active {{
      max-width: none;
      box-shadow: none;
      border-radius: 0;
      margin: 0;
    }}
  }}
</style>
</head>
<body>
<div class="layout">
  <div class="sidebar">
    <img src="{logo_uri}" alt="한국마사회" class="sidebar-logo">
    <h2>지원자 목록 ({count}명)</h2>
    {sidebar_items}
  </div>
  <div class="content">
    <div class="placeholder" id="placeholder">좌측 목록에서 면접번호를 선택하세요.</div>
    {panels}
  </div>
</div>
<script>
  function showReport(examNo) {{
    document.querySelectorAll('.report-panel').forEach(function(p) {{
      p.classList.toggle('active', p.id === 'panel-' + examNo);
    }});
    document.querySelectorAll('.sidebar .item').forEach(function(b) {{
      b.classList.toggle('active', b.dataset.examNo === examNo);
    }});
    var ph = document.getElementById('placeholder');
    if (ph) ph.style.display = 'none';
  }}

  document.addEventListener('DOMContentLoaded', function() {{
    var first = document.querySelector('.sidebar .item');
    if (first) showReport(first.dataset.examNo);
  }});
</script>
</body>
</html>
"""


def build_combined_report(all_reports):
    """all_reports: list of (exam_no, field, intro_qa, main_questions, summary_map)"""
    sidebar_items = []
    panels = []

    for exam_no, field, intro_qa, main_questions, summary_map in all_reports:
        exam_no_str = html.escape(str(exam_no))
        field_str = html.escape(str(field or ""))

        sidebar_items.append(f"""
        <button class="item" data-exam-no="{exam_no_str}" onclick="showReport('{exam_no_str}')">
          {exam_no_str}
          <span class="field">{field_str}</span>
        </button>""")

        body = render_report_body(exam_no, field, intro_qa, main_questions, summary_map)
        panels.append(f"""
        <div class="report-panel" id="panel-{exam_no_str}">
        {body}
        </div>""")

    return COMBINED_TEMPLATE.format(
        title=html.escape(REPORT_TITLE),
        common_style=COMMON_STYLE,
        logo_uri=get_logo_data_uri(),
        count=len(all_reports),
        sidebar_items="\n".join(sidebar_items),
        panels="\n".join(panels),
    )


# ======================================================================
# main
# ======================================================================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    intro_data = load_intro(INTRO_PATH)
    interview_data = load_interview_questions(INTERVIEW_PATH)
    summary_data = load_summaries(INTERVIEW_PATH)

    if not summary_data:
        print("[안내] '문항요약' 시트가 없어 자기소개서 원문에서 자동 추출한 요약을 사용합니다.")
        print("       GPT 요약을 쓰려면 STEP 1 셀을 먼저 실행하세요.\n")

    all_exam_nos = sorted(set(intro_data.keys()) | set(interview_data.keys()), key=str)
    print(f"총 {len(all_exam_nos)}명 처리")

    all_reports = []
    for exam_no in all_exam_nos:
        intro = intro_data.get(exam_no, {"모집분야": "", "qa": []})
        interview = interview_data.get(
            exam_no, {"모집분야": intro.get("모집분야", ""), "main_questions": []}
        )

        field = intro.get("모집분야") or interview.get("모집분야") or ""
        intro_qa = intro["qa"]
        main_questions = interview["main_questions"]
        summary_map = summary_data.get(exam_no, {})

        report_html = build_individual_report(
            exam_no, field, intro_qa, main_questions, summary_map
        )
        out_path = os.path.join(OUTPUT_DIR, f"면접리포트_{exam_no}.html")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(report_html)
        print(f"  - {out_path}")

        all_reports.append((exam_no, field, intro_qa, main_questions, summary_map))

    combined_html = build_combined_report(all_reports)
    combined_path = os.path.join(OUTPUT_DIR, "전체리포트.html")
    with open(combined_path, "w", encoding="utf-8") as f:
        f.write(combined_html)
    print(f"  - {combined_path}  (좌측 목록 선택 통합 리포트)")

    print("완료")


main()


 STEP 2 · 개별 면접리포트 HTML 생성


① 자기소개서 엑셀 파일 경로
 >  "C:\Users\LG\Desktop\마사회 전임직\한국마사회 전임직(산업안전관리) 자기소개서_보훈특별.xlsx"
② 면접질문 생성결과 엑셀 파일 경로 (STEP 1 결과)
 >  "C:\Users\LG\Desktop\마사회 전임직\한국마사회 전임직(산업안전관리) 자기소개서_보훈특별_면접질문생성결과.xlsx"
③ 리포트를 저장할 폴더
  [Enter = C:\Users\LG\Desktop\마사회 전임직\개인별리포트]
 >  
④ 리포트 제목
  [Enter = 2026년 한국마사회 전임직·위촉직채용 개별면접질문]
 >  2026 한국마사회 전임직(산업안전관리) 보훈특별채용
⑤ 로고 이미지 경로 (한국마사회 로고를 쓰려면 그냥 Enter)
 >  "C:\Users\LG\Desktop\마사회 전임직\logo.svg"



총 3명 처리
  - C:\Users\LG\Desktop\마사회 전임직\개인별리포트\면접리포트_D1-001.html
  - C:\Users\LG\Desktop\마사회 전임직\개인별리포트\면접리포트_D1-002.html
  - C:\Users\LG\Desktop\마사회 전임직\개인별리포트\면접리포트_D1-003.html
  - C:\Users\LG\Desktop\마사회 전임직\개인별리포트\전체리포트.html  (좌측 목록 선택 통합 리포트)
완료
